# Pinned controlled evaluation rerun

This notebook creates a fresh Colab checkout and reruns the locked 20-item controlled evaluation under the exact pinned runtime. Select a **T4 GPU** before starting. Add a read-capable `HF_TOKEN` to Colab Secrets and enable notebook access.

Required upload: `controlled_training_bundle-4.zip`.


In [ ]:
# 1. Confirm the required GPU.
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU")
print("GPU:", torch.cuda.get_device_name(0))
if "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This gate requires a Tesla T4")


In [ ]:
# 2. Clone a fresh checkout and select the exact implementation commit.
REPOSITORY_COMMIT = "7aded20b49aed6f21bf4f9cee7700d3b856fc599"
REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_ROOT = "/content/VLM-pinned"
PROJECT_ROOT = f"{REPOSITORY_ROOT}/gi_vqa_research"

from pathlib import Path
import shutil
import subprocess

checkout = Path(REPOSITORY_ROOT)
if checkout.exists():
    shutil.rmtree(checkout)
subprocess.run(["git", "clone", REPOSITORY_URL, REPOSITORY_ROOT], check=True)
subprocess.run(["git", "checkout", REPOSITORY_COMMIT], cwd=checkout, check=True)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=checkout, check=True,
    capture_output=True, text=True,
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=checkout, check=True,
    capture_output=True, text=True,
).stdout
print("Commit:", observed)
print("Checkout clean:", not bool(status))
if observed != REPOSITORY_COMMIT or status:
    raise RuntimeError("Fresh checkout verification failed")
%cd /content/VLM-pinned/gi_vqa_research


In [ ]:
# 3. Install the exact project GPU dependencies and compatible fsspec.
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-e", ".[gpu]", "fsspec==2024.12.0",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
print("Dependency installation completed")


In [ ]:
# 4. Verify every runtime package enforced by the evaluation gate.
import importlib.metadata as metadata
import sys
import torch

expected = {
    "accelerate": "1.9.0",
    "bitsandbytes": "0.47.0",
    "datasets": "3.3.2",
    "huggingface-hub": "0.34.3",
    "ms-swift": "3.7.0",
    "numpy": "2.0.2",
    "peft": "0.16.0",
    "Pillow": "11.3.0",
    "PyYAML": "6.0.2",
    "sentencepiece": "0.2.0",
    "transformers": "4.55.0",
    "wandb": "0.21.0",
}
mismatches = {}
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
for package, required in expected.items():
    observed = metadata.version(package)
    print(f"{package}: {observed}")
    if observed != required:
        mismatches[package] = {"expected": required, "observed": observed}
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Python 3.11 is required")
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("A CUDA Tesla T4 is required")
if not str(torch.__version__).startswith("2.6.0"):
    raise RuntimeError(f"Expected PyTorch 2.6.0, observed {torch.__version__}")
if mismatches:
    raise RuntimeError(f"Package mismatches: {mismatches}")
print("\nPinned runtime PASS")


In [ ]:
# 5. Authenticate without printing or persisting the token in an artifact.
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download, login, whoami

hf_token = userdata.get("HF_TOKEN")
if not isinstance(hf_token, str) or not hf_token.strip():
    raise RuntimeError("HF_TOKEN is missing or notebook access is disabled")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
identity = whoami(token=hf_token)
print("Authenticated as:", identity.get("name") or identity.get("fullname"))
path = hf_hub_download(
    repo_id="google/paligemma-3b-pt-224",
    filename="config.json",
    revision="35e4f46485b4d07967e7e9935bc3786aad50687c",
    token=hf_token,
)
print("PaliGemma access PASS:", path)
del hf_token


In [ ]:
# 6. Upload controlled_training_bundle-4.zip when prompted.
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded))
if "controlled_training_bundle-4.zip" not in uploaded:
    raise RuntimeError("Upload must be named controlled_training_bundle-4.zip")


In [ ]:
# 7. Extract and validate the training bundle layout.
from pathlib import Path
import zipfile

project = Path(PROJECT_ROOT)
archive = project / "controlled_training_bundle-4.zip"
destination = project / "controlled_training_bundle-4"
destination.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        target = (destination / member.filename).resolve()
        target.relative_to(destination.resolve())
    bundle.extractall(destination)
required = [
    destination / "bundle_manifest.json",
    destination / "controlled_training_report.json",
    destination / "adapters/paired_image/adapter_model.safetensors",
    destination / "adapters/constant_image/adapter_model.safetensors",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError(f"Bundle is incomplete: {missing}")
print("Training bundle PASS")


In [ ]:
# 8. Confirm ignored inputs did not dirty the Git checkout.
import subprocess

status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_ROOT,
    check=True, capture_output=True, text=True,
).stdout
print(status or "Checkout clean")
if status:
    raise RuntimeError("Checkout is dirty; do not run the gate")


In [ ]:
# 9. Run the fresh pinned evaluation. This may take some time.
import os
import subprocess
import sys

environment = os.environ.copy()
environment["PYTHONPATH"] = "src"
command = [
    sys.executable, "-m", "gi_vqa.controlled_evaluation_runner",
    "--project-root", ".",
    "--training-bundle", "controlled_training_bundle-4",
    "--run-dir", "runs/controlled_evaluation_pilot",
    "--expected-commit", REPOSITORY_COMMIT,
    "--require-clean-git",
    "--required-gpu-substring", "T4",
]
subprocess.run(command, cwd=PROJECT_ROOT, env=environment, check=True)
print("Controlled evaluation process completed")


In [ ]:
# 10. Inspect the final status and condition metrics.
import json
from pathlib import Path

report_path = Path(PROJECT_ROOT) / "runs/controlled_evaluation_pilot/controlled_evaluation_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Status:", report["status"])
print("Test accessed:", report["test_partition_accessed"])
print("Commit:", report["repository"]["commit"])
print("Runtime packages:", json.dumps(report["runtime"]["packages"], indent=2))
for condition, result in report["conditions"].items():
    print(f"\n{condition}")
    print(json.dumps(result["metrics"], indent=2))
print("\nComparisons against base")
print(json.dumps(report["comparisons"], indent=2))
if report["status"] != "PASS" or report["test_partition_accessed"] is not False:
    raise RuntimeError("Controlled evaluation did not produce an acceptable PASS")


In [ ]:
# 11. Package the new pinned evidence separately from the provisional run.
from pathlib import Path
import shutil

project = Path(PROJECT_ROOT)
run_directory = project / "runs/controlled_evaluation_pilot"
archive_base = project / "controlled_evaluation_pinned_bundle"
if not run_directory.is_dir():
    raise FileNotFoundError(run_directory)
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=project,
    base_dir="runs/controlled_evaluation_pilot",
)
print("Created:", archive_path)


In [ ]:
# 12. Download the pinned evidence ZIP to your computer.
from google.colab import files

files.download(
    "/content/VLM-pinned/gi_vqa_research/controlled_evaluation_pinned_bundle.zip"
)
